# 41. The Terminal Concession Bidding Problem

## Tier 1: Integer Programming Formulation

### Goal
Formulate and solve a multi-objective integer programming model for terminal concession bidding that captures the strategic decision-making of both bidders and the port authority.

### Key assumptions
- Only one operator can win the concession
- Bidders must meet minimum qualification thresholds
- Financial commitments are limited by bidder capacity
- Performance guarantees must meet minimum requirements

### Approach (step-by-step)
1. Define sets and parameters for bidders, criteria, and periods
2. Formulate decision variables for selection and commitments
3. Create multi-objective functions for revenue maximization
4. Implement constraints for qualification and financial capacity
5. Solve using mixed-integer programming with pulp solver
6. Perform sensitivity analysis on key parameters

### What to look for in the results
- Optimal bidder selection based on weighted scoring
- Financial commitment optimization across periods
- Sensitivity of results to qualification thresholds
- Trade-offs between revenue and performance objectives

### Concrete example (from the source)
3 bidders (MTC, GPH, ACS) competing for terminal concession evaluated on 4 criteria:
- Financial Capacity (weight=0.3)
- Technical Experience (weight=0.25) 
- Revenue Commitment (weight=0.25)
- Performance Guarantees (weight=0.2)

### Why this Tier exists vs earlier Tiers
This is the foundational Tier that provides the mathematical optimization baseline with provable optimality guarantees. Unlike heuristics or metaheuristics, the integer programming approach ensures globally optimal solutions for the concession bidding problem.

### Pros / Cons vs other approaches
**Pros:**
- Guaranteed optimal solutions
- Clear mathematical formulation
- Handles constraints explicitly
- Provides sensitivity analysis capabilities

**Cons:**
- Computationally intensive for large instances
- Requires precise parameter estimation
- Limited scalability with many bidders/criteria
- May oversimplify complex strategic interactions

### When to use this Tier
- Small to medium-sized bidding scenarios (≤10 bidders)
- When optimality guarantees are required
- Regulatory environments requiring transparent methodology
- Strategic planning and sensitivity analysis

In [1]:
# Import required libraries for integer programming
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pulp import *
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [2]:
# Define data structures for terminal concession bidding
from dataclasses import dataclass
from typing import Dict, List, Tuple

@dataclass
class Bidder:
    """Represents a bidding operator with capabilities and commitments"""
    name: str
    financial_capacity: float  # Maximum upfront payment (million €)
    technical_experience: int   # Score 0-100
    revenue_commitment: float  # Annual revenue guarantee (million €)
    performance_guarantee: int # Score 0-100
    employment_commitment: int # Number of direct jobs
    automation_investment: float # Automation investment (million €)

@dataclass
class Criterion:
    """Represents an evaluation criterion with weight"""
    name: str
    weight: float
    min_score: int = 0  # Minimum score to qualify

@dataclass
class ConcessionProblem:
    """Defines the terminal concession bidding problem"""
    bidders: List[Bidder]
    criteria: List[Criterion]
    min_qualification_threshold: float
    concession_duration: int  # years
    annual_throughput_target: float  # million TEU

In [ ]:
# Create the concrete example from the source material
def create_concession_example():
    """Create the terminal concession bidding example"""
    
    # Define bidders with their characteristics
    bidders = [
        Bidder("MTC", 85, 90, 80, 95, 280, 45),   # Mediterranean Terminal Company
        Bidder("GPH", 95, 85, 90, 85, 320, 65),   # Global Port Holdings  
        Bidder("ACS", 80, 95, 85, 90, 295, 55),   # Atlantic Container Services
    ]
    
    # Define evaluation criteria with weights
    criteria = [
        Criterion("Financial Capacity", 0.30, 70),
        Criterion("Technical Experience", 0.25, 75),
        Criterion("Revenue Commitment", 0.25, 70),
        Criterion("Performance Guarantees", 0.20, 80),
    ]
    
    # Create problem instance
    problem = ConcessionProblem(
        bidders=bidders,
        criteria=criteria,
        min_qualification_threshold=85.0,
        concession_duration=25,
        annual_throughput_target=1.5
    )
    
    return problem

# Create the problem instance
concession_problem = create_concession_example()
print(f"Created concession problem with {len(concession_problem.bidders)} bidders and {len(concession_problem.criteria)} criteria")

In [ ]:
# Calculate weighted scores for each bidder
def calculate_weighted_scores(problem: ConcessionProblem) -> pd.DataFrame:
    """Calculate weighted scores for all bidders"""
    
    scores_data = []
    
    for bidder in problem.bidders:
        # Get individual criterion scores
        criterion_scores = {
            "Financial Capacity": bidder.financial_capacity,
            "Technical Experience": bidder.technical_experience,
            "Revenue Commitment": bidder.revenue_commitment,
            "Performance Guarantees": bidder.performance_guarantee,
        }
        
        # Calculate weighted total score
        total_score = 0
        qualified = True
        
        for criterion in problem.criteria:
            score = criterion_scores[criterion.name]
            weighted_score = score * criterion.weight
            total_score += weighted_score
            
            # Check qualification
            if score < criterion.min_score:
                qualified = False
        
        scores_data.append({
            'Bidder': bidder.name,
            'Financial Capacity': criterion_scores["Financial Capacity"],
            'Technical Experience': criterion_scores["Technical Experience"],
            'Revenue Commitment': criterion_scores["Revenue Commitment"],
            'Performance Guarantees': criterion_scores["Performance Guarantees"],
            'Total Score': total_score,
            'Qualified': qualified
        })
    
    return pd.DataFrame(scores_data)

# Calculate and display scores
scores_df = calculate_weighted_scores(concession_problem)
print("=== Terminal Concession Bidding Evaluation ===")
print(scores_df.to_string(index=False))
print(f"\nMinimum qualification threshold: {concession_problem.min_qualification_threshold}")

In [ ]:
# Implement integer programming formulation
def solve_concession_bidding_mip(problem: ConcessionProblem) -> Dict:
    """Solve terminal concession bidding using mixed-integer programming"""
    
    # Create the optimization problem
    model = LpProblem("Terminal_Concession_Bidding", LpMaximize)
    
    # Decision variables
    # x_i: 1 if bidder i is selected, 0 otherwise
    x = {bidder.name: LpVariable(f"select_{bidder.name}", cat='Binary') 
         for bidder in problem.bidders}
    
    # z_i: actual payment commitment by bidder i
    z = {bidder.name: LpVariable(f"payment_{bidder.name}", lowBound=0, 
                                cat='Continuous') 
         for bidder in problem.bidders}
    
    # Get criterion scores for optimization
    scores_df = calculate_weighted_scores(problem)
    
    # Objective: Maximize total weighted score of selected bidder
    model += lpSum(scores_df.loc[scores_df['Bidder'] == bidder.name, 'Total Score'].iloc[0] * x[bidder.name] 
                   for bidder in problem.bidders)
    
    # Constraint: Only one winner
    model += lpSum(x[bidder.name] for bidder in problem.bidders) == 1, "Single_Winner"
    
    # Constraint: Qualification requirements
    for bidder in problem.bidders:
        qualified = scores_df.loc[scores_df['Bidder'] == bidder.name, 'Qualified'].iloc[0]
        if not qualified:
            model += x[bidder.name] == 0, f"Disqualified_{bidder.name}"
    
    # Constraint: Financial capacity
    for bidder in problem.bidders:
        model += z[bidder.name] <= bidder.financial_capacity * x[bidder.name], \
                f"Financial_Capacity_{bidder.name}"
    
    # Constraint: Minimum qualification threshold
    for bidder in problem.bidders:
        total_score = scores_df.loc[scores_df['Bidder'] == bidder.name, 'Total Score'].iloc[0]
        model += total_score * x[bidder.name] >= problem.min_qualification_threshold * x[bidder.name], \
                f"Qualification_Threshold_{bidder.name}"
    
    # Solve the model
    print("Solving MIP formulation...")
    model.solve(PULP_CBC_CMD(msg=False))
    
    # Extract results
    results = {
        'status': LpStatus[model.status],
        'winner': None,
        'winning_score': 0,
        'payment_commitment': 0,
        'all_scores': scores_df
    }
    
    if model.status == LpStatusOptimal:
        for bidder in problem.bidders:
            if x[bidder.name].value() == 1:
                results['winner'] = bidder.name
                results['winning_score'] = scores_df.loc[scores_df['Bidder'] == bidder.name, 'Total Score'].iloc[0]
                results['payment_commitment'] = z[bidder.name].value()
                break
    
    return results, model

# Solve the concession bidding problem
results, model = solve_concession_bidding_mip(concession_problem)
print(f"\nOptimization Status: {results['status']}")
if results['winner']:
    print(f"Winner: {results['winner']}")
    print(f"Winning Score: {results['winning_score']:.2f}")
    print(f"Payment Commitment: €{results['payment_commitment']:.1f} million")

In [ ]:
# Create comprehensive visualization of results
def visualize_concession_results(problem: ConcessionProblem, results: Dict):
    """Create comprehensive visualization of concession bidding results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Terminal Concession Bidding Analysis', fontsize=16, fontweight='bold')
    
    # 1. Bidder Comparison - Radar Chart
    ax1 = axes[0, 0]
    categories = ['Financial Capacity', 'Technical Experience', 'Revenue Commitment', 'Performance Guarantees']
    
    # Create radar chart data
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]  # Complete the circle
    
    for bidder in problem.bidders:
        values = [
            bidder.financial_capacity,
            bidder.technical_experience,
            bidder.revenue_commitment,
            bidder.performance_guarantee
        ]
        values += values[:1]  # Complete the circle
        
        ax1.plot(angles, values, 'o-', linewidth=2, label=bidder.name)
        ax1.fill(angles, values, alpha=0.25)
    
    ax1.set_xticks(angles[:-1])
    ax1.set_xticklabels(categories)
    ax1.set_ylim(0, 100)
    ax1.set_title('Bidder Capability Comparison')
    ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    ax1.grid(True)
    
    # 2. Weighted Score Breakdown
    ax2 = axes[0, 1]
    scores_df = results['all_scores']
    
    x_pos = np.arange(len(scores_df))
    width = 0.6
    
    bars = ax2.bar(x_pos, scores_df['Total Score'], width, 
                   color=['#2E86AB', '#A23B72', '#F18F01'][:len(scores_df)])
    
    # Add qualification threshold line
    ax2.axhline(y=problem.min_qualification_threshold, color='red', 
                linestyle='--', label=f'Minimum Threshold ({problem.min_qualification_threshold})')
    
    # Highlight winner
    if results['winner']:
        winner_idx = scores_df[scores_df['Bidder'] == results['winner']].index[0]
        bars[winner_idx].set_edgecolor('gold')
        bars[winner_idx].set_linewidth(3)
    
    ax2.set_xlabel('Bidders')
    ax2.set_ylabel('Weighted Score')
    ax2.set_title('Weighted Evaluation Scores')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(scores_df['Bidder'], rotation=45)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Financial Commitment Analysis
    ax3 = axes[1, 0]
    
    financial_data = [(b.name, b.financial_capacity) for b in problem.bidders]
    names, capacities = zip(*financial_data)
    
    colors = ['#FF6B6B' if name == results['winner'] else '#4ECDC4' for name in names]
    bars3 = ax3.bar(names, capacities, color=colors, alpha=0.7)
    
    # Add winner's commitment
    if results['winner'] and results['payment_commitment'] > 0:
        winner_idx = names.index(results['winner'])
        ax3.bar(names[winner_idx], results['payment_commitment'], 
                color='gold', alpha=0.9, label=f'Commitment: €{results["payment_commitment"]:.1f}M')
    
    ax3.set_xlabel('Bidders')
    ax3.set_ylabel('Financial Capacity (Million €)')
    ax3.set_title('Financial Capacity Analysis')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Criterion Weight Importance
    ax4 = axes[1, 1]
    
    criterion_names = [c.name for c in problem.criteria]
    weights = [c.weight for c in problem.criteria]
    
    colors4 = plt.cm.Set3(np.linspace(0, 1, len(criterion_names)))
    wedges, texts, autotexts = ax4.pie(weights, labels=criterion_names, autopct='%1.1f%%', 
                                        colors=colors4, startangle=90)
    ax4.set_title('Evaluation Criteria Weights')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Visualize the results
fig = visualize_concession_results(concession_problem, results)

In [ ]:
# Sensitivity analysis on qualification threshold
def sensitivity_analysis(problem: ConcessionProblem) -> pd.DataFrame:
    """Perform sensitivity analysis on qualification threshold"""
    
    thresholds = np.arange(70, 96, 2.5)  # From 70 to 95
    sensitivity_results = []
    
    # Store original threshold
    original_threshold = problem.min_qualification_threshold
    
    for threshold in thresholds:
        # Update threshold
        problem.min_qualification_threshold = threshold
        
        # Solve with new threshold
        try:
            results, _ = solve_concession_bidding_mip(problem)
            
            if results['winner']:
                sensitivity_results.append({
                    'Threshold': threshold,
                    'Winner': results['winner'],
                    'Winning Score': results['winning_score'],
                    'Qualified Bidders': len(results['all_scores'][results['all_scores']['Qualified']])
                })
            else:
                sensitivity_results.append({
                    'Threshold': threshold,
                    'Winner': 'No Solution',
                    'Winning Score': 0,
                    'Qualified Bidders': 0
                })
        except Exception as e:
            sensitivity_results.append({
                'Threshold': threshold,
                'Winner': 'Error',
                'Winning Score': 0,
                'Qualified Bidders': 0
            })
    
    # Restore original threshold
    problem.min_qualification_threshold = original_threshold
    
    return pd.DataFrame(sensitivity_results)

# Perform sensitivity analysis
sensitivity_df = sensitivity_analysis(concession_problem)
print("=== Sensitivity Analysis on Qualification Threshold ===")
print(sensitivity_df.to_string(index=False))

# Visualize sensitivity results
plt.figure(figsize=(12, 6))

# Plot 1: Winner vs Threshold
plt.subplot(1, 2, 1)
thresholds = sensitivity_df['Threshold']
winners = sensitivity_df['Winner']

# Assign colors to different winners
unique_winners = winners.unique()
colors = plt.cm.Set2(np.linspace(0, 1, len(unique_winners)))
winner_colors = {winner: color for winner, color in zip(unique_winners, colors)}

for winner in unique_winners:
    mask = winners == winner
    plt.plot(thresholds[mask], [1] * sum(mask), 'o-', 
             color=winner_colors[winner], label=winner, linewidth=2, markersize=8)

plt.xlabel('Qualification Threshold')
plt.ylabel('Winner Selection')
plt.title('Winner vs Qualification Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 2)

# Plot 2: Qualified Bidders vs Threshold
plt.subplot(1, 2, 2)
plt.plot(thresholds, sensitivity_df['Qualified Bidders'], 's-', 
         color='#E74C3C', linewidth=2, markersize=8)
plt.xlabel('Qualification Threshold')
plt.ylabel('Number of Qualified Bidders')
plt.title('Qualified Bidders vs Threshold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comprehensive summary and conclusions
def generate_summary_report(problem: ConcessionProblem, results: Dict, sensitivity_df: pd.DataFrame):
    """Generate comprehensive summary report"""
    
    print("\n" + "="*80)
    print("TERMINAL CONCESSION BIDDING - INTEGER PROGRAMMING ANALYSIS")
    print("="*80)
    
    print(f"\nProblem Configuration:")
    print(f"- Number of Bidders: {len(problem.bidders)}")
    print(f"- Number of Criteria: {len(problem.criteria)}")
    print(f"- Concession Duration: {problem.concession_duration} years")
    print(f"- Annual Throughput Target: {problem.annual_throughput_target} million TEU")
    print(f"- Minimum Qualification Threshold: {problem.min_qualification_threshold}")
    
    print(f"\nOptimization Results:")
    print(f"- Status: {results['status']}")
    if results['winner']:
        print(f"- Winning Bidder: {results['winner']}")
        print(f"- Winning Score: {results['winning_score']:.2f}")
        print(f"- Payment Commitment: €{results['payment_commitment']:.1f} million")
        
        # Get winner details
        winner = next(b for b in problem.bidders if b.name == results['winner'])
        print(f"\nWinning Bidder Profile:")
        print(f"- Financial Capacity: €{winner.financial_capacity} million")
        print(f"- Technical Experience: {winner.technical_experience}/100")
        print(f"- Revenue Commitment: €{winner.revenue_commitment} million/year")
        print(f"- Performance Guarantee: {winner.performance_guarantee}/100")
        print(f"- Employment Commitment: {winner.employment_commitment} jobs")
        print(f"- Automation Investment: €{winner.automation_investment} million")
    
    print(f"\nAll Bidders Evaluation:")
    scores_df = results['all_scores']
    for _, row in scores_df.iterrows():
        status = "✓ Qualified" if row['Qualified'] else "✗ Disqualified"
        winner_mark = " 🏆" if row['Bidder'] == results['winner'] else ""
        print(f"- {row['Bidder']}: {row['Total Score']:.2f} ({status}){winner_mark}")
    
    print(f"\nSensitivity Analysis Insights:")
    threshold_changes = sensitivity_df[sensitivity_df['Winner'] != sensitivity_df.iloc[0]['Winner']]
    if len(threshold_changes) > 0:
        print(f"- Winner changes at threshold values: {sorted(threshold_changes['Threshold'].unique())}")
        print(f"- Most stable winner: {sensitivity_df['Winner'].mode().iloc[0]}")
    else:
        print(f"- Winner selection is stable across all threshold values")
    
    max_qualified = sensitivity_df['Qualified Bidders'].max()
    min_qualified = sensitivity_df['Qualified Bidders'].min()
    print(f"- Qualified bidders range: {min_qualified} to {max_qualified}")
    
    print(f"\nInteger Programming Advantages:")
    print(f"✓ Guaranteed optimal solution")
    print(f"✓ Explicit constraint handling")
    print(f"✓ Sensitivity analysis capability")
    print(f"✓ Transparent decision process")
    
    print(f"\nComputational Performance:")
    print(f"- Model size: {len(problem.bidders)} binary variables")
    print(f"- Constraints: {len(problem.bidders) * 3 + 1}")
    print(f"- Solution time: < 1 second (small instance)")
    
    print(f"\nPractical Applications:")
    print(f"- Regulatory compliance and transparency")
    print(f"- Strategic planning and scenario analysis")
    print(f"- Stakeholder communication and justification")
    print(f"- Risk assessment and sensitivity testing")
    
    print("\n" + "="*80)

# Generate comprehensive summary
generate_summary_report(concession_problem, results, sensitivity_df)